# Recombination with GRG

**Note:** To add sample nodes (e.g., offspring from recombination), use grgl from the `grg_modify_improve` branch:

```bash
git clone -b grg_modify_improve --recursive https://github.com/aprilweilab/grgl.git
cd grgl && python setup.py bdist_wheel && pip install --force-reinstall dist/*.whl
```

The `MutableGRG.set_samples(sample_nodes)` API lets you add new sample nodes by passing the full list of sample node IDs (including the new offspring).

In [34]:
import numpy as np
import bisect

NEGATIVE_NODE_IDS = []  
def get_breakpoints(N, p=0.1, min_breakpoints=3):
    """
    Return breakpoint positions. Ensures at least min_breakpoints in interior
    so recombination is less likely to be entire interval from one parent.
    """
    bp = np.where(np.random.binomial(1, p, N))[0]
    # Ensure at least one interior breakpoint to avoid entire-interval inheritance
    interior = np.arange(1, N)
    while len(bp) < min_breakpoints and len(interior) >= min_breakpoints:
        bp = np.unique(np.concatenate([bp, np.random.choice(interior, min(min_breakpoints, len(interior)), replace=False)]))
        bp = np.sort(bp)
    print(f"Generated breakpoints: {bp}")
    return bp

def recombination_intervals(h1, h2, N):
    """
    Returns list of segments: [(source_parent_id, end_coord), ...]
    """
    bp = get_breakpoints(N)
    start = np.random.binomial(1, 0.5, 1)[0]
    parents = [h1, h2]
    
    segments = []
    for i, K in enumerate(bp):
        segments.append((parents[(start + i) % 2], K))
    segments.append((parents[(start + len(bp)) % 2], N))
    return segments

In [73]:
from pygrgl import MutableGRG, Mutation, save_grg, grg_to_cyto_json
import pygrgl

def create_simple_grg():
    grg = MutableGRG(4, 1, True) 
    
    # Create 4 internal nodes (4, 5, 6, 7)
    grg.make_node()  # node 4 (has m6, m7)
    grg.make_node()  # node 5 (has m8)
    grg.make_node()  # node 6 (has m9)
    grg.make_node()  # node 7 (has m10, m11)
    
    # Add edges (parent -> child) 
    grg.connect(6, 0)  # 6 -> 0
    grg.connect(6, 5)  # 6 -> 5
    grg.connect(7, 0)  # 7 -> 0
    grg.connect(7, 4)  # 7 -> 4
    grg.connect(5, 1)  # 5 -> 1
    grg.connect(5, 4)  # 5 -> 4
    grg.connect(4, 2)  # 4 -> 2
    grg.connect(4, 3)  # 4 -> 3
    
    # Add mutations (m1-m11)
    grg.add_mutation(Mutation(0, "A", "G"), 1)   # m1 on node 1
    grg.add_mutation(Mutation(1, "C", "T"), 0)   # m2 on node 0
    grg.add_mutation(Mutation(2, "G", "A"), 0)   # m3 on node 0
    grg.add_mutation(Mutation(3, "T", "C"), 2)   # m4 on node 2
    grg.add_mutation(Mutation(4, "A", "T"), 3)   # m5 on node 3
    grg.add_mutation(Mutation(5, "C", "G"), 4)   # m6 on node 4
    grg.add_mutation(Mutation(6, "G", "C"), 4)   # m7 on node 4
    grg.add_mutation(Mutation(7, "T", "A"), 5)   # m8 on node 5
    grg.add_mutation(Mutation(8, "A", "C"), 6)   # m9 on node 6
    grg.add_mutation(Mutation(9, "C", "A"), 7)  # m10 on node 7
    grg.add_mutation(Mutation(10, "G", "T"), 7)  # m11 on node 7
    
    return grg

# Create and save the GRG
simple_grg = create_simple_grg()


print("=== Simple GRG Created ===")
print(f"Nodes: {simple_grg.num_nodes}")
print(f"Edges: {simple_grg.num_edges}")
print(f"Mutations: {simple_grg.num_mutations}")
print(f"Samples: {simple_grg.get_sample_nodes()}")
print()

# Display structure
print("Node details:")
for node_id in range(simple_grg.num_nodes):
    parents = simple_grg.get_up_edges(node_id)
    children = simple_grg.get_down_edges(node_id)
    muts = simple_grg.get_mutations_for_node(node_id)
    mut_names = [f"m{m+1}" for m in muts]  # Just show mutation names
    is_sample = "[SAMPLE]" if simple_grg.is_sample(node_id) else ""
    print(f"  Node {node_id}: parents={parents}, children={children}, muts={mut_names} {is_sample}")

# Save to file
save_grg(simple_grg, "simple_example.grg")
print()
print("Saved to simple_example.grg")

cyto_data = grg_to_cyto_json(simple_grg)
print()
print("Cytoscape JSON representation:")
print(f"  Nodes: {len(cyto_data['nodes'])}")
print(f"  Edges: {len(cyto_data['edges'])}")
for node in cyto_data['nodes']:
    print(f"    {node['data']}")
for edge in cyto_data['edges']:
    print(f"    {edge['data']}")

=== Simple GRG Created ===
Nodes: 8
Edges: 8
Mutations: 11
Samples: [0, 1, 2, 3]

Node details:
  Node 0: parents=[6, 7], children=[], muts=['m2', 'm3'] [SAMPLE]
  Node 1: parents=[5], children=[], muts=['m1'] [SAMPLE]
  Node 2: parents=[4], children=[], muts=['m4'] [SAMPLE]
  Node 3: parents=[4], children=[], muts=['m5'] [SAMPLE]
  Node 4: parents=[7, 5], children=[2, 3], muts=['m6', 'm7'] 
  Node 5: parents=[6], children=[1, 4], muts=['m8'] 
  Node 6: parents=[], children=[0, 5], muts=['m9'] 
  Node 7: parents=[], children=[0, 4], muts=['m10', 'm11'] 

Saved to simple_example.grg

Cytoscape JSON representation:
  Nodes: 8
  Edges: 8
    {'id': 'n0', 'label': 'id=0, S, mutations({1, 2})', 'is_sample': 'True'}
    {'id': 'n1', 'label': 'id=1, S, mutations({0})', 'is_sample': 'True'}
    {'id': 'n2', 'label': 'id=2, S, mutations({3})', 'is_sample': 'True'}
    {'id': 'n3', 'label': 'id=3, S, mutations({4})', 'is_sample': 'True'}
    {'id': 'n4', 'label': 'id=4, mutations({5, 6})', 'is_s

In [36]:
GRG_FILE = "simple_example.grg" 

In [68]:
class NonDuplicationRecombination:
    """
    Non-duplication GRG recombination algorithm.

    Optimizations vs. the recursive version:
    - Iterative DFS for `_recurse_attach` and `_get_node_and_ancestor_span`
      (avoids Python's recursion limit on deep GRGs).
    - O(1) offspring reverse lookup via a dict alongside NEGATIVE_NODE_IDS.
    - Per-recombiner cache for `get_up_edges()`.
    - Generation-counter arrays replace per-call Python sets for
      `visited` (per DFS) and `connected` (per offspring).
    - Optional deferred sample updates via `defer_sample_updates`.
    - Hot-loop micro-optimizations: cache-hit fast paths for mutation
      range and ancestral coverage are inlined into `_recurse_attach`,
      with class attributes hoisted into local variables (CPython
      optimizes LOAD_FAST faster than LOAD_ATTR).
    - `_extract_bubble` no longer invalidates the caches of node_id's
      parents; bubble extraction does not affect their span or ancestral
      coverage, so invalidation just thrashes high-traffic cache entries.

    Audit instrumentation (Audit 1):
    - `self.audit` accumulates per-decision-case counts plus primitives
      (extract_bubble, connect, make_node, recombine, recurse_attach).
    - `audit_check()` asserts the algorithm/implementation identities:
        extract_bubble_calls == sum(bubble cases)
        total connects       == 2*extract_bubble_calls + (firing direct attaches)
        make_node_calls      == recombine_calls + extract_bubble_calls
      Failure of any identity isolates a bug to a specific branch.
    - `audit_summary()` pretty-prints the case histogram and ratios.
    - `audit_reset()` zeroes counters (call between independent runs).
    """

    debug_mode = True

    def __init__(self, grg):
        self.grg = grg
        self.genome_length = grg.bp_range[1]
        self.original_bp_range = grg.bp_range
        self.NEGATIVE_NODE_IDS = []
        self._negative_node_index = {}
        self._modified_nodes = set()
        self._pending_bubbles = []
        self._pending_sample_removals = set()

        self.defer_sample_updates = False

        self._mutation_cache = {}
        self._pos_cache = {}
        self._up_edges_cache = {}

        self.span_cache = [False] * self.grg.num_nodes
        self.anc_cov_cache = [False] * self.grg.num_nodes

        self._visited_gen = [0] * self.grg.num_nodes
        self._connected_gen = [0] * self.grg.num_nodes
        self._gen_visited = 0
        self._gen_connected = 0

        # ----- Audit 1: case-counter instrumentation -----
        # Every visit in _recurse_attach falls into exactly one of the
        # 13 decision cells below (9 matrix cells + 4 root variants).
        # Skip-counters are informational (not in the matrix).
        self.audit = {
            # row "no relevant" (Mu cap I = empty)
            'pruning':                  0,  # has_no_relevant + ancestral_disjoint
            'pruning_root':             0,  # has_no_relevant + Iu is None
            'path_compression':         0,  # has_no_relevant + full_coverage
            'decomposition':            0,  # has_no_relevant + partial overlap
            # row "all covered" (Mu subset I)
            'direct_attach':            0,  # has_all_relevant + full_coverage
            'direct_attach_root':       0,  # has_all_relevant + Iu is None
            'bubble_strip':             0,  # has_all_relevant + ancestral_disjoint
            'bubble_split':             0,  # has_all_relevant + partial overlap
            'direct_attach_dup':        0,  # connected_gen guard skipped a duplicate edge
            # row "partial relevant"
            'bubble_fill':              0,  # has_partial_relevant + full_coverage
            'bubble_strip_partial':     0,  # has_partial_relevant + ancestral_disjoint
            'bubble_split_partial':     0,  # has_partial_relevant + partial overlap
            'bubble_strip_partial_rt':  0,  # has_partial_relevant + Iu is None
            # informational skips (don't change matrix counts)
            'skip_empty_interval':      0,  # L >= R at top of loop
            'skip_already_visited':     0,  # gen_v guard fired
            'skip_empty_trim':          0,  # newL >= newR after trim, no recursion fired
            # primitives that should reconcile against case sums
            'visits':                   0,  # entries past both skip guards
            'extract_bubble_calls':     0,
            'connect_calls_in_attach':  0,  # connects fired inside _recurse_attach (Direct Attach)
            'connect_calls_in_extract': 0,  # connects fired inside _extract_bubble (always 2/call)
            'make_node_calls':          0,  # offspring + bubble nodes created
            'recombine_calls':          0,  # recombine() + recombine_multi() entries
            'recurse_attach_calls':     0,
        }

    # ------------------------------------------------------------------
    # Per-node array growth
    # ------------------------------------------------------------------

    def _grow_node_arrays(self, node_id):
        target = node_id + 1
        n = len(self.span_cache)
        if n < target:
            pad = target - n
            self.span_cache.extend([False] * pad)
            self.anc_cov_cache.extend([False] * pad)
        n = len(self._visited_gen)
        if n < target:
            pad = target - n
            self._visited_gen.extend([0] * pad)
            self._connected_gen.extend([0] * pad)

    def _sync_to_grg(self):
        n = self.grg.num_nodes
        if n > 0:
            self._grow_node_arrays(n - 1)

    # ------------------------------------------------------------------
    # Cached graph access
    # ------------------------------------------------------------------

    def _get_up_edges_cached(self, node_id):
        cached = self._up_edges_cache.get(node_id)
        if cached is None:
            cached = list(self.grg.get_up_edges(node_id))
            self._up_edges_cache[node_id] = cached
        return cached

    # ------------------------------------------------------------------
    # Mutation caches
    # ------------------------------------------------------------------

    def _get_node_mutations(self, node_id):
        if node_id not in self._mutation_cache:
            mut_ids = self.grg.get_mutations_for_node(node_id, allow_sort=False)
            mutations = []
            for mut_id in mut_ids:
                mut = self.grg.get_mutation_by_id(mut_id)
                mutations.append((mut_id, mut.position))
            combined = sorted(mutations, key=lambda x: x[1])
            self._mutation_cache[node_id] = combined
            self._pos_cache[node_id] = [m[1] for m in combined]
        return self._mutation_cache[node_id]

    def _get_mutation_range(self, node_id, L, R):
        self._get_node_mutations(node_id)
        positions = self._pos_cache[node_id]
        if not positions:
            return 0, 0
        return bisect.bisect_left(positions, L), bisect.bisect_left(positions, R)

    # ------------------------------------------------------------------
    # Iterative span / ancestral coverage
    # ------------------------------------------------------------------

    def _get_node_and_ancestor_span(self, node_id):
        if self.span_cache[node_id] is not False:
            return self.span_cache[node_id]

        stack = [(node_id, False)]
        scheduled = {node_id}

        while stack:
            nid, processed = stack.pop()

            if processed:
                min_p, max_p = float('inf'), float('-inf')
                node_muts = self._get_node_mutations(nid)
                if node_muts:
                    min_p = node_muts[0][1]
                    max_p = node_muts[-1][1]
                for parent in self._get_up_edges_cached(nid):
                    anc = self.span_cache[parent]
                    if anc:
                        if anc[0] < min_p: min_p = anc[0]
                        if anc[1] > max_p: max_p = anc[1]
                self.span_cache[nid] = None if min_p == float('inf') else (min_p, max_p)
                scheduled.discard(nid)
            else:
                stack.append((nid, True))
                for parent in self._get_up_edges_cached(nid):
                    if self.span_cache[parent] is False and parent not in scheduled:
                        stack.append((parent, False))
                        scheduled.add(parent)

        return self.span_cache[node_id]

    def _get_ancestral_coverage(self, node_id):
        if self.anc_cov_cache[node_id] is not False:
            return self.anc_cov_cache[node_id]

        parents = self._get_up_edges_cached(node_id)
        if not parents:
            self.anc_cov_cache[node_id] = None
            return None

        min_pos, max_pos = float('inf'), float('-inf')
        for parent in parents:
            p_span = self._get_node_and_ancestor_span(parent)
            if p_span:
                if p_span[0] < min_pos: min_pos = p_span[0]
                if p_span[1] > max_pos: max_pos = p_span[1]

        result = None if min_pos == float('inf') else (min_pos, max_pos + 1)
        self.anc_cov_cache[node_id] = result
        return result

    # ------------------------------------------------------------------
    # Bubble extraction
    # ------------------------------------------------------------------

    def _extract_bubble(self, node_id, relevant_mut_ids, offspring_id, interval):
        bubble_id = self.grg.make_node()
        self.audit['make_node_calls'] += 1
        self._grow_node_arrays(bubble_id)

        self.grg.connect(bubble_id, node_id)
        self.grg.connect(bubble_id, -offspring_id)
        self.audit['connect_calls_in_extract'] += 2
        self.audit['extract_bubble_calls'] += 1

        # node_id just gained a new up-edge; drop its cached up-edges list.
        self._up_edges_cache.pop(node_id, None)

        self._pending_bubbles.append({
            'node_id': node_id,
            'bubble_id': bubble_id,
            'relevant_mut_ids': relevant_mut_ids,
        })

        # Track only nodes whose own caches actually become stale:
        # node_id loses mutations (mutation/pos/anc_cov are stale) and the
        # new bubble_id needs a clean slot. node_id's parents are NOT
        # invalidated -- their span and anc_cov depend only on themselves
        # and their own ancestors, neither of which changes here.
        self._modified_nodes.add(node_id)
        self._modified_nodes.add(bubble_id)

        return bubble_id

    # ------------------------------------------------------------------
    # Iterative recurse-attach (hot path)
    # ------------------------------------------------------------------

    def _recurse_attach(self, root_id, offspring_id, L0, R0):
        self._gen_visited += 1
        gen_v = self._gen_visited
        gen_c = self._gen_connected

        if self.debug_mode:
            print(f"[recurse_attach] enter root={root_id} interval=[{L0}, {R0}) offspring={-offspring_id} gen_v={gen_v}")

        # Hoist attributes / globals into locals for the inner loop.
        # CPython's LOAD_FAST is faster than LOAD_ATTR; over hundreds of
        # thousands of iterations these adds up.
        visited_gen = self._visited_gen
        connected_gen = self._connected_gen
        pos_cache = self._pos_cache
        mutation_cache = self._mutation_cache
        anc_cov_cache = self.anc_cov_cache
        bisect_left = bisect.bisect_left
        get_node_mutations = self._get_node_mutations
        get_ancestral_coverage = self._get_ancestral_coverage
        get_up_edges_cached = self._get_up_edges_cached
        extract_bubble = self._extract_bubble
        grg_connect = self.grg.connect
        pending_sample_removals_add = self._pending_sample_removals.add
        audit = self.audit  # hoisted for inner-loop speed

        audit['recurse_attach_calls'] += 1

        neg_offspring = -offspring_id
        stack = [(root_id, L0, R0)]
        stack_append = stack.append
        stack_pop = stack.pop

        while stack:
            node_id, L, R = stack_pop()

            if L >= R:
                audit['skip_empty_interval'] += 1
                if self.debug_mode:
                    print(f"  visit node={node_id}: skip (interval [{L}, {R}) is empty)")
                continue
            if visited_gen[node_id] == gen_v:
                audit['skip_already_visited'] += 1
                if self.debug_mode:
                    print(f"  visit node={node_id} interval=[{L}, {R}): skip (already visited this segment)")
                continue
            visited_gen[node_id] = gen_v
            audit['visits'] += 1

            # Inlined _get_mutation_range + _get_node_mutations cache-hit path.
            positions = pos_cache.get(node_id)
            if positions is None:
                get_node_mutations(node_id)              # populates both caches
                positions = pos_cache[node_id]

            if positions:
                left = bisect_left(positions, L)
                right = bisect_left(positions, R)
            else:
                left = right = 0

            num_rel = right - left
            num_all = len(positions)

            has_all_relevant = num_all > 0 and num_rel == num_all
            has_no_relevant = num_rel == 0
            has_partial_relevant = num_rel > 0 and num_rel < num_all

            # Inlined _get_ancestral_coverage cache-hit path.
            Iu = anc_cov_cache[node_id]
            if Iu is False:
                Iu = get_ancestral_coverage(node_id)

            if self.debug_mode:
                rel_kind = ("all" if has_all_relevant
                            else "none" if has_no_relevant
                            else "partial")
                print(f"  visit node={node_id} interval=[{L}, {R}) "
                      f"Mu_rel={num_rel}/{num_all} ({rel_kind}) Iu={Iu}")

            if Iu is None:
                # Root node (no parents) -> only the node's own mutations matter.
                if has_all_relevant:
                    if connected_gen[node_id] != gen_c:
                        if self.debug_mode:
                            print(f"    -> Direct Attach (root): connect node {node_id} to offspring {neg_offspring}, stop")
                        grg_connect(node_id, neg_offspring)
                        connected_gen[node_id] = gen_c
                        pending_sample_removals_add(node_id)
                        audit['direct_attach_root'] += 1
                        audit['connect_calls_in_attach'] += 1
                    else:
                        if self.debug_mode:
                            print(f"    -> Direct Attach (root): node {node_id} already connected to offspring this call, stop")
                        audit['direct_attach_dup'] += 1
                elif has_partial_relevant:
                    rel_mut_ids = [m[0] for m in mutation_cache[node_id][left:right]]
                    if self.debug_mode:
                        print(f"    -> Bubble & Strip Partial (root): bubble muts {rel_mut_ids} from node {node_id}, stop")
                    bubble_id = extract_bubble(node_id, rel_mut_ids, offspring_id, (L, R))
                    connected_gen[bubble_id] = gen_c
                    audit['bubble_strip_partial_rt'] += 1
                else:
                    if self.debug_mode:
                        print(f"    -> Pruning (root): no relevant muts on root {node_id}, stop")
                    audit['pruning_root'] += 1
                continue

            Iu0 = Iu[0]
            Iu1 = Iu[1]
            ancestral_disjoint = R <= Iu0 or L >= Iu1
            full_coverage = Iu0 >= L and Iu1 <= R

            if has_all_relevant and full_coverage:
                # Direct Attach (Mu subset I, Iu subset I): edge from u carries
                # everything we need.
                if connected_gen[node_id] != gen_c:
                    if self.debug_mode:
                        print(f"    -> Direct Attach: connect node {node_id} to offspring {neg_offspring}, stop")
                    grg_connect(node_id, neg_offspring)
                    connected_gen[node_id] = gen_c
                    pending_sample_removals_add(node_id)
                    audit['direct_attach'] += 1
                    audit['connect_calls_in_attach'] += 1
                else:
                    if self.debug_mode:
                        print(f"    -> Direct Attach: node {node_id} already connected to offspring this call, stop")
                    audit['direct_attach_dup'] += 1
                continue

            if has_no_relevant and ancestral_disjoint:
                # Pruning (Mu cap I = empty, Iu cap I = empty): dead end.
                if self.debug_mode:
                    print(f"    -> Pruning: dead end at node {node_id}, stop")
                audit['pruning'] += 1
                continue

            if has_no_relevant:
                # Path Compression (full coverage) or Decomposition (partial overlap):
                # node u contributes no mutations but ancestry might.
                newL = L if L > Iu0 else Iu0
                newR = R if R < Iu1 else Iu1
                if full_coverage:
                    audit['path_compression'] += 1
                    case = "Path Compression"
                else:
                    audit['decomposition'] += 1
                    case = "Decomposition"
                if newL >= newR:
                    audit['skip_empty_trim'] += 1
                    if self.debug_mode:
                        print(f"    -> {case}: trimmed interval [{newL}, {newR}) empty at node {node_id}, stop")
                    continue
                if self.debug_mode:
                    parents = get_up_edges_cached(node_id)
                    print(f"    -> {case}: bypass node {node_id}, recurse parents {parents} with [{newL}, {newR})")
                for parent in reversed(get_up_edges_cached(node_id)):
                    stack_append((parent, newL, newR))
                continue

            # Bubble cases: Mu has at least one mutation in I, but the lineage at
            # node u is not "All Covered + Full" (already handled above).
            rel_mut_ids = [m[0] for m in mutation_cache[node_id][left:right]]
            bubble_id = extract_bubble(node_id, rel_mut_ids, offspring_id, (L, R))
            connected_gen[bubble_id] = gen_c

            # Classify exactly one of the 5 non-root bubble cells.
            if has_all_relevant:
                if ancestral_disjoint:
                    audit['bubble_strip'] += 1
                    case = "Bubble & Strip"
                else:
                    audit['bubble_split'] += 1
                    case = "Bubble & Split"
            else:  # has_partial_relevant
                if full_coverage:
                    audit['bubble_fill'] += 1
                    case = "Bubble & Fill"
                elif ancestral_disjoint:
                    audit['bubble_strip_partial'] += 1
                    case = "Bubble & Strip Partial"
                else:
                    audit['bubble_split_partial'] += 1
                    case = "Bubble & Split Partial"

            if self.debug_mode:
                print(f"    -> {case}: bubble {bubble_id} captures muts {rel_mut_ids} from node {node_id}")

            if ancestral_disjoint:
                if self.debug_mode:
                    print(f"      stop (ancestors disjoint from [{L}, {R}))")
                continue

            newL = L if L > Iu0 else Iu0
            newR = R if R < Iu1 else Iu1
            if newL >= newR:
                audit['skip_empty_trim'] += 1
                if self.debug_mode:
                    print(f"      stop (trimmed interval [{newL}, {newR}) empty)")
                continue
            if self.debug_mode:
                parents = get_up_edges_cached(node_id)
                print(f"      recurse parents {parents} with [{newL}, {newR})")
            for parent in reversed(get_up_edges_cached(node_id)):
                stack_append((parent, newL, newR))

    # ------------------------------------------------------------------
    # Apply deferred work, evict caches
    # ------------------------------------------------------------------

    def _apply_pending_bubbles(self):
        if self.debug_mode and self._pending_bubbles:
            print(f"[apply_pending_bubbles] applying {len(self._pending_bubbles)} bubble(s)")
        for bubble_op in self._pending_bubbles:
            node_id = bubble_op['node_id']
            bubble_id = bubble_op['bubble_id']
            if self.debug_mode:
                print(f"  bubble {bubble_id}: move muts {bubble_op['relevant_mut_ids']} from node {node_id} -> bubble {bubble_id}")
            for mut_id in bubble_op['relevant_mut_ids']:
                mut = self.grg.get_mutation_by_id(mut_id)
                self.grg.add_mutation(mut, bubble_id)
                self.grg.remove_mutation(mut_id, node_id)
        self._pending_bubbles.clear()

        if not self.defer_sample_updates:
            self.flush_sample_updates()

    def flush_sample_updates(self):
        if self._pending_sample_removals:
            if self.debug_mode:
                print(f"[flush_sample_updates] removing {sorted(self._pending_sample_removals)} from samples (gained new bubble parents)")
            current = set(self.grg.get_sample_nodes())
            current.difference_update(self._pending_sample_removals)
            self.grg.set_samples(list(current))
            self._pending_sample_removals.clear()

    def _clear_modified_caches(self):
        for node_id in self._modified_nodes:
            self._mutation_cache.pop(node_id, None)
            self._pos_cache.pop(node_id, None)
            if node_id < len(self.span_cache):
                self.span_cache[node_id] = False
                self.anc_cov_cache[node_id] = False
        self._modified_nodes.clear()

    # ------------------------------------------------------------------
    # Public entry points
    # ------------------------------------------------------------------

    def _register_offspring(self, offspring_id):
        idx = self._negative_node_index.get(offspring_id)
        if idx is None:
            idx = len(self.NEGATIVE_NODE_IDS)
            self._negative_node_index[offspring_id] = idx
            self.NEGATIVE_NODE_IDS.append(offspring_id)
        return -(idx + 1)

    def recombine(self, haplotype_A, haplotype_B, breakpoint):
        self.audit['recombine_calls'] += 1
        self._pending_bubbles.clear()
        self._sync_to_grg()
        offspring_id = self.grg.make_node(negative=True)
        self.audit['make_node_calls'] += 1
        self._grow_node_arrays(self.grg.num_nodes - 1)
        self._gen_connected += 1

        if self.debug_mode:
            print("=" * 60)
            print(f"[recombine] offspring={-offspring_id} hA={haplotype_A} hB={haplotype_B} bp={breakpoint}")
            print(f"[recombine] segment 1: parent={haplotype_A} interval=[0, {breakpoint})")
        self._recurse_attach(haplotype_A, offspring_id, 0, breakpoint)
        if self.debug_mode:
            print(f"[recombine] segment 2: parent={haplotype_B} interval=[{breakpoint}, {self.genome_length})")
        self._recurse_attach(haplotype_B, offspring_id, breakpoint, self.genome_length)

        self._apply_pending_bubbles()
        self._clear_modified_caches()

        return self._register_offspring(offspring_id)

    def recombine_multi(self, segments):
        self.audit['recombine_calls'] += 1
        self._pending_bubbles.clear()
        self._sync_to_grg()
        offspring_id = self.grg.make_node(negative=True)
        self.audit['make_node_calls'] += 1
        self._grow_node_arrays(self.grg.num_nodes - 1)
        self._gen_connected += 1

        if self.debug_mode:
            print("=" * 60)
            print(f"[recombine_multi] offspring={-offspring_id} segments={segments}")

        start = 0
        for parent_id, end in segments:
            if end > start:
                if self.debug_mode:
                    print(f"[recombine_multi] segment: parent={parent_id} interval=[{start}, {end})")
                self._recurse_attach(parent_id, offspring_id, start, end)
            start = end

        self._apply_pending_bubbles()
        self._clear_modified_caches()

        return self._register_offspring(offspring_id)

    # ------------------------------------------------------------------
    # Audit 1: invariant checks and reporting
    # ------------------------------------------------------------------

    def audit_check(self, raise_on_fail=True):
        """
        Verify the implementation matches the algorithm spec on cumulative
        counts. Returns a dict describing each invariant and whether it
        passed; raises AssertionError on first failure if raise_on_fail.

        Invariants:
            (1) extract_bubble_calls == sum of the 6 bubble-case counters.
                A failure here means a bubble was created outside the matrix
                (or a matrix branch ran without calling _extract_bubble).
            (2) total connects == 2*extract_bubble_calls + (firing direct
                attaches). A failure means connect() was invoked outside
                the two paths the algorithm prescribes.
            (3) make_node_calls == recombine_calls + extract_bubble_calls.
                A failure means a node was created outside the offspring
                or bubble paths.
        """
        a = self.audit

        bubble_cases = (a['bubble_strip'] + a['bubble_split'] +
                        a['bubble_fill'] + a['bubble_strip_partial'] +
                        a['bubble_split_partial'] + a['bubble_strip_partial_rt'])
        # "direct_attach_dup" entries did NOT call connect (gen_c guard fired),
        # so they are excluded from the connect identity below.
        direct_firing = a['direct_attach'] + a['direct_attach_root']
        total_connects = a['connect_calls_in_attach'] + a['connect_calls_in_extract']
        expected_connects = 2 * a['extract_bubble_calls'] + direct_firing
        expected_make_nodes = a['recombine_calls'] + a['extract_bubble_calls']

        results = {
            'bubble_identity': {
                'lhs': a['extract_bubble_calls'],
                'rhs': bubble_cases,
                'pass': a['extract_bubble_calls'] == bubble_cases,
                'desc': 'extract_bubble_calls == sum of 6 bubble-case counters',
            },
            'connect_identity': {
                'lhs': total_connects,
                'rhs': expected_connects,
                'pass': total_connects == expected_connects,
                'desc': 'total connect calls == 2*bubbles + firing direct attaches',
            },
            'make_node_identity': {
                'lhs': a['make_node_calls'],
                'rhs': expected_make_nodes,
                'pass': a['make_node_calls'] == expected_make_nodes,
                'desc': 'make_node_calls == recombine_calls + bubbles',
            },
        }

        if raise_on_fail:
            for name, r in results.items():
                if not r['pass']:
                    raise AssertionError(
                        f"audit_check FAIL [{name}]: {r['desc']} -- "
                        f"lhs={r['lhs']}, rhs={r['rhs']}, delta={r['lhs'] - r['rhs']}"
                    )

        return results

    def audit_summary(self):
        """Pretty-print the case histogram and derived ratios."""
        a = self.audit
        bubble_cases = (a['bubble_strip'] + a['bubble_split'] +
                        a['bubble_fill'] + a['bubble_strip_partial'] +
                        a['bubble_split_partial'] + a['bubble_strip_partial_rt'])
        direct_firing = a['direct_attach'] + a['direct_attach_root']
        total_decisions = (
            a['pruning'] + a['pruning_root']
            + a['path_compression'] + a['decomposition']
            + a['direct_attach'] + a['direct_attach_root'] + a['direct_attach_dup']
            + bubble_cases
        )

        def pct(n):
            return (100.0 * n / total_decisions) if total_decisions else 0.0

        lines = []
        lines.append("=" * 60)
        lines.append("AUDIT 1 -- decision case histogram")
        lines.append("=" * 60)
        lines.append(f"{'case':<30s} {'count':>12s}  {'%':>6s}")
        lines.append("-" * 60)
        for k in ('pruning', 'pruning_root',
                  'path_compression', 'decomposition',
                  'direct_attach', 'direct_attach_root', 'direct_attach_dup',
                  'bubble_strip', 'bubble_split',
                  'bubble_fill',
                  'bubble_strip_partial', 'bubble_split_partial',
                  'bubble_strip_partial_rt'):
            lines.append(f"{k:<30s} {a[k]:>12d}  {pct(a[k]):>5.1f}%")
        lines.append("-" * 60)
        lines.append(f"{'TOTAL DECISIONS':<30s} {total_decisions:>12d}")
        lines.append("")
        lines.append(f"{'visits (post-skip-guards)':<30s} {a['visits']:>12d}")
        lines.append(f"{'skip_empty_interval':<30s} {a['skip_empty_interval']:>12d}")
        lines.append(f"{'skip_already_visited':<30s} {a['skip_already_visited']:>12d}")
        lines.append(f"{'skip_empty_trim':<30s} {a['skip_empty_trim']:>12d}")
        lines.append("")
        lines.append("Primitives:")
        lines.append(f"{'recombine_calls':<30s} {a['recombine_calls']:>12d}")
        lines.append(f"{'recurse_attach_calls':<30s} {a['recurse_attach_calls']:>12d}")
        lines.append(f"{'extract_bubble_calls':<30s} {a['extract_bubble_calls']:>12d}")
        lines.append(f"{'make_node_calls':<30s} {a['make_node_calls']:>12d}")
        lines.append(f"{'connect (in _recurse_attach)':<30s} {a['connect_calls_in_attach']:>12d}")
        lines.append(f"{'connect (in _extract_bubble)':<30s} {a['connect_calls_in_extract']:>12d}")
        lines.append("")
        if a['recombine_calls']:
            lines.append("Per-recombine averages:")
            lines.append(f"  bubbles / offspring     {a['extract_bubble_calls']/a['recombine_calls']:>10.3f}")
            lines.append(f"  visits / offspring      {a['visits']/a['recombine_calls']:>10.3f}")
            lines.append(f"  segments / offspring    {a['recurse_attach_calls']/a['recombine_calls']:>10.3f}")
            lines.append(f"  edges added / offspring {(2*bubble_cases + direct_firing)/a['recombine_calls']:>10.3f}")
        lines.append("")
        lines.append("Identities (via audit_check):")
        results = self.audit_check(raise_on_fail=False)
        for name, r in results.items():
            mark = "OK" if r['pass'] else "FAIL"
            lines.append(f"  [{mark}] {r['desc']}: lhs={r['lhs']} rhs={r['rhs']}")
        lines.append("=" * 60)
        print("\n".join(lines))

    def audit_reset(self):
        """Zero all audit counters. Use between independent benchmark runs."""
        for k in self.audit:
            self.audit[k] = 0

In [38]:
import pygrgl
from pygrgl import load_mutable_grg, grg_to_cyto_json
import re

try:
    from pygrgl.display import grg_to_cyto, DAG_STYLE
    from ipycytoscape import CytoscapeWidget
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False
    DAG_STYLE = None
    CytoscapeWidget = None

if 'NEGATIVE_NODE_IDS' not in globals():
    NEGATIVE_NODE_IDS = []

def display_grg(grg, title="GRG", negative_node_ids=None):
    """Display GRG - uses widget if available, otherwise text."""
    neg_ids = negative_node_ids if negative_node_ids is not None else NEGATIVE_NODE_IDS
    cyto = grg_to_cyto_json(grg, start_from=grg.get_root_nodes())

    for n in cyto['nodes']:
        node_id = int(n['data']['id'][1:])
        if node_id in neg_ids:
            disp = -(neg_ids.index(node_id) + 1)
            n['data']['label'] = n['data']['label'].replace(f'id={node_id}', f'id={disp}', 1)
        
        # Record positions of mutations for this node to display in label
        positions = []
        for mut_id in grg.get_mutations_for_node(node_id):
            mut = grg.get_mutation_by_id(mut_id)
            positions.append(mut.position)

        pattern = r"mutations\(\{.*?\}\)"
        positions_str = ", ".join(str(n) for n in positions)
        # Build the replacement string using your new contents
        replacement = f"mutations({{{positions_str}}})"

        # Replace the whole mutations({ ... }) with the new one
        new_str = re.sub(pattern, replacement, n['data']['label'])
        n['data']['label'] = new_str
        
    if WIDGETS_AVAILABLE:
        try:
            widget = CytoscapeWidget()
            widget.graph.add_graph_from_json(cyto, directed=True)
            widget.set_style(DAG_STYLE)
            widget.set_layout(name="dagre")
            display(widget)
            return
        except Exception as e:
            print(f"Widget display failed: {e}")
    
    print(f"\n{'='*50}")
    print(f" {title}")
    print(f"{'='*50}")
    print(f"Nodes: {len(cyto['nodes'])}, Edges: {len(cyto['edges'])}")
    print("\nNodes:")
    for n in cyto['nodes']:
        d = n['data']
        marker = "[S]" if d.get('is_sample') == 'True' else "   "
        print(f"  {marker} {d['id']}: {d['label']}")
    print("\nEdges (parent → child):")
    for e in cyto['edges']:
        src, tgt = e['data']['source'], e['data']['target']
        try:
            tgt_id = int(tgt[1:])
            tgt_display = f'n{-(neg_ids.index(tgt_id) + 1)}' if tgt_id in neg_ids else tgt
        except ValueError:
            tgt_display = tgt
        print(f"      {src} → {tgt_display}")
    print(f"{'='*50}\n")

grg = create_simple_grg()

def print_grg_state(grg, title="GRG State"):
    """Helper to print GRG state."""
    print(f"=== {title} ===")
    print(f"Nodes: {grg.num_nodes}, Edges: {grg.num_edges}, Mutations: {grg.num_mutations}")
    print(f"Samples: {grg.get_sample_nodes()}")
    print(f"Genome range: {grg.bp_range}")
    print()
    all_nodes = pygrgl.get_topo_order(grg, pygrgl.TraversalDirection.DOWN, grg.get_root_nodes())
    for node_id in all_nodes:
        display_id = -(NEGATIVE_NODE_IDS.index(node_id) + 1) if node_id in NEGATIVE_NODE_IDS else node_id
        up = grg.get_up_edges(node_id)
        down = grg.get_down_edges(node_id)
        muts = grg.get_mutations_for_node(node_id)
        mut_names = [f"m{m+1}" for m in muts]  # Just show mutation names
        is_sample = " [SAMPLE]" if grg.is_sample(node_id) else ""
        print(f"  Node {display_id}: parents={up}, children={down}, muts={mut_names}{is_sample}")

# Show initial state
#print_grg_state(grg, "BEFORE Recombination")
display_grg(grg, "BEFORE Recombination")

CytoscapeWidget(cytoscape_layout={'name': 'dagre'}, cytoscape_style=[{'selector': 'node', 'style': {'font-fami…

In [39]:
recomb = NonDuplicationRecombination(grg)

haplotype_A = 2  
haplotype_B = 3  
breakpoint = 3 
segments = []
segments.append((haplotype_A, breakpoint))
segments.append((haplotype_B, grg.bp_range[1]))

print("=" * 60)
print("RECOMBINATION")
print("=" * 60)
print(f"Haplotype A: {haplotype_A}")
print(f"Haplotype B: {haplotype_B}")
print(f"Breakpoint: {breakpoint}")
print()
print(f"Offspring inherits [0, {breakpoint}) from haplotype {haplotype_A}")
print(f"Offspring inherits [{breakpoint}, {grg.bp_range[1]}) from haplotype {haplotype_B}")
print()

# Perform recombination
offspring_id = recomb.recombine(haplotype_A, haplotype_B, breakpoint)

try:
    raw_id = recomb.NEGATIVE_NODE_IDS[abs(offspring_id) - 1]
    current_samples = list(grg.get_sample_nodes())
    grg.set_samples(current_samples + [raw_id])
except AttributeError:
    pass

print(f"Created offspring node: {offspring_id}")
print()

# Show state after recombination
print_grg_state(grg, "AFTER Recombination")
display_grg(grg, "AFTER Recombination")

RECOMBINATION
Haplotype A: 2
Haplotype B: 3
Breakpoint: 3

Offspring inherits [0, 3) from haplotype 2
Offspring inherits [3, 11) from haplotype 3

Created offspring node: -1

=== AFTER Recombination ===
Nodes: 9, Edges: 10, Mutations: 11
Samples: [0, 1, 2, 8]
Genome range: (0, 11)

  Node 7: parents=[], children=[5, 4], muts=['m10', 'm11']
  Node 6: parents=[], children=[0, 5], muts=['m9']
  Node 5: parents=[6, 7], children=[1, 4], muts=['m8']
  Node 4: parents=[7, 5], children=[1, 2, 3], muts=['m6', 'm7']
  Node 3: parents=[4], children=[8], muts=['m5']
  Node 8: parents=[3], children=[], muts=[] [SAMPLE]
  Node 2: parents=[4], children=[], muts=['m4'] [SAMPLE]
  Node 1: parents=[5, 4], children=[], muts=['m1'] [SAMPLE]
  Node 0: parents=[6], children=[], muts=['m2', 'm3'] [SAMPLE]


CytoscapeWidget(cytoscape_layout={'name': 'dagre'}, cytoscape_style=[{'selector': 'node', 'style': {'font-fami…

In [89]:
def verify_offspring_mutations(recomb, offspring_id, haplotype_A, haplotype_B, segments):
    """
    Verify that offspring has correct mutations based on recombination.
    
    Expected mutations:
    - From haplotype_A: all ancestral mutations with position < breakpoint
    - From haplotype_B: all ancestral mutations with position >= breakpoint
    """
    def get_all_ancestral_mutations(node_id, visited=None):
        """Get all mutation IDs from node and its ancestors."""
        if visited is None:
            visited = set()
        if node_id in visited:
            return []
        visited.add(node_id)
        
        mutations = list(recomb.grg.get_mutations_for_node(node_id))
        
        for parent in recomb.grg.get_up_edges(node_id):
            mutations.extend(get_all_ancestral_mutations(parent, visited))
        
        return mutations
    
    def mut_name(mut_id):
        return f"m{mut_id + 1}"
    
    def get_position(mut_id):
        return recomb.grg.get_mutation_by_id(mut_id).position
    
    genome_length = recomb.grg.bp_range[1]
    
    # Get ancestral mutations for both haplotypes
    muts_A = get_all_ancestral_mutations(haplotype_A)
    muts_B = get_all_ancestral_mutations(haplotype_B)

    # Expected mutations for offspring based on segments
    expected_from_A = []
    expected_from_B = []

    # Determine expected mutations based on segments and their source haplotypes
    start = 0
    for parent, end in segments:
        if parent == haplotype_A:
            for m in muts_A:
                if get_position(m) >= start and get_position(m) < end:
                    expected_from_A.append(m)
        else:
            for m in muts_B:
                if get_position(m) >= start and get_position(m) < end:
                    expected_from_B.append(m)
        start = end
    
    expected_muts = set(expected_from_A + expected_from_B)
    expected_from_A.sort()
    expected_from_B.sort()
    
    # Get actual offspring mutations (traversing ancestry)
    grg_node_id = recomb.NEGATIVE_NODE_IDS[abs(offspring_id) - 1] if offspring_id < 0 else offspring_id
    actual_muts = set(get_all_ancestral_mutations(grg_node_id))

    muts_A.sort()
    muts_B.sort()
    
    if recomb.debug_mode:
        print("=== Mutation Verification ===")
        print(f"Haplotype A (node {haplotype_A}) mutations: {[get_position(m) for m in muts_A]}")
        print(f"Haplotype B (node {haplotype_B}) mutations: {[get_position(m) for m in muts_B]}")
        print()
        print(f"Expected from A: {[get_position(m) for m in expected_from_A]}")
        print(f"Expected from B: {[get_position(m) for m in expected_from_B]}")
        print()
        print(f"Expected: {sorted([get_position(m) for m in expected_muts])}")
        print(f"Actual:   {sorted([get_position(m) for m in actual_muts])}")
        print()
    
    if expected_muts == actual_muts:
        print("✓ Offspring mutations are CORRECT!")
        return True
    else:
        missing = expected_muts - actual_muts
        extra = actual_muts - expected_muts
        if missing:
            print(f"✗ Missing: {[get_position(m) for m in missing]}")
        if extra:
            print(f"✗ Extra: {[get_position(m) for m in extra]}")
        return False

# Verify the recombination
print("=== Verification ===")
breakpoints = []
breakpoints.append(breakpoint)
recombs = NonDuplicationRecombination(grg)
verify_offspring_mutations(recombs, offspring_id, haplotype_A, haplotype_B, segments)

=== Verification ===


IndexError: list index out of range

In [78]:
def generate_offspring(recomb, h1, h2, num_offspring, N=None):
    """
    Generate a recombined offspring from two parent haplotypes.
    
    Uses the recombination_intervals function to generate random
    crossover breakpoints, then applies non-duplication recombination.
    
    Args:
        grg: MutableGRG instance
        h1: First parent haplotype node ID
        h2: Second parent haplotype node ID
        N: Genome length (defaults to grg.bp_range[1])
        
    Returns:
        Tuple of (offspring_node_id, segments)
    """
    if N is None:
        N = grg.bp_range[1]
        print(f"Using genome length from GRG: {N}")
    
    offspring_ids = []
    for i in range(num_offspring):
        # Get recombination segments
        segments = recombination_intervals(h1, h2, N)
    
        # Perform recombination
        offspring_id = recomb.recombine_multi(segments)
        raw_id = recomb.NEGATIVE_NODE_IDS[abs(offspring_id) - 1]
        offspring_ids.append(raw_id)

        if recomb.debug_mode:
            print()
            print(f"Generated offspring node: {offspring_id}")
            print(f"Samples (including offspring): {grg.get_sample_nodes()}")
            print(f"Segments inherited: {segments}")
            print()

            print("Segment breakdown:")
            start = 0
            for parent, end in segments:
                print(f"  [{start}, {end}): from parent {parent}")
                start = end
            
            print("")
        verify_offspring_mutations(recomb, offspring_id, h1, h2, segments)

    # try:
    #     current_samples = list(grg.get_sample_nodes())
    #     raw_id = NEGATIVE_NODE_IDS[abs(offspring_id) - 1]
    #     grg.set_samples(current_samples + [raw_id])
    # except AttributeError:
    #     pass  
    
    return segments, offspring_ids



print("=== Random Recombination Example ===")
grg_test = create_simple_grg() 

print(f"Loading: {GRG_FILE}")
print(f"Genome length: {grg_test.bp_range[1]}")
print(f"Initial samples: {grg_test.get_sample_nodes()}")
print()

# display_grg(grg_test, "BEFORE random recombination")

genome = grg_test.bp_range
generations = 2

# Create recombination handler
recomb = NonDuplicationRecombination(grg_test)

for gen in range(generations):
    print(f"\n=== Generation {gen+1} ===")

    parent_indices = np.arange(len(grg_test.get_sample_nodes()))
    np.random.shuffle(parent_indices)
    new_offspring = []

    samples = grg_test.get_sample_nodes()
    shuffled_samples = np.random.shuffle(samples)

    #offspring, segs = generate_offspring(grg_test, h1=0, num_offspring=2, h2=3)
    for i in range(0, len(samples), 2):

        p1 = samples[i]
        p2 = samples[i+1]

        display_grg(grg_test, f"BEFORE random recombination {i+1}")

        print(f"Selected parents for offspring {i+1}: h1={p1}, h2={p2}")
        segments, offspring_ids = generate_offspring(recomb, p1, p2, num_offspring=2, N=genome[1])
        new_offspring.extend(offspring_ids)

    new_offspring.sort()
    grg_test.set_samples(new_offspring)
    

# print()
# print(f"Generated offspring node: {offspring}")
# print(f"Samples (including offspring): {grg_test.get_sample_nodes()}")
# print(f"Segments inherited: {segs}")
# print()

# print("Segment breakdown:")
# start = 0
# for parent, end in segs:
#     print(f"  [{start}, {end}): from parent {parent}")
#     start = end

# Show after state
display_grg(grg_test, "AFTER random recombination")
recomb.audit_summary()


=== Random Recombination Example ===
Loading: simple_example.grg
Genome length: 11
Initial samples: [0, 1, 2, 3]


=== Generation 1 ===


CytoscapeWidget(cytoscape_layout={'name': 'dagre'}, cytoscape_style=[{'selector': 'node', 'style': {'font-fami…

Selected parents for offspring 1: h1=3, h2=2
Generated breakpoints: [0 8 9]
[recombine_multi] offspring=-8 segments=[(2, np.int64(0)), (3, np.int64(8)), (2, np.int64(9)), (3, 11)]
[recombine_multi] segment: parent=3 interval=[0, 8)
[recurse_attach] enter root=3 interval=[0, 8) offspring=-8 gen_v=1
  visit node=3 interval=[0, 8) Mu_rel=1/1 (all) Iu=(5, 11)
    -> Bubble & Split: bubble 9 captures muts [4] from node 3
      recurse parents [4, 9] with [5, 8)
  visit node=4 interval=[5, 8) Mu_rel=2/2 (all) Iu=(7, 11)
    -> Bubble & Split: bubble 10 captures muts [5, 6] from node 4
      recurse parents [7, 5, 10] with [7, 8)
  visit node=7 interval=[7, 8) Mu_rel=0/2 (none) Iu=None
    -> Pruning (root): no relevant muts on root 7, stop
  visit node=5 interval=[7, 8) Mu_rel=1/1 (all) Iu=(8, 9)
    -> Bubble & Strip: bubble 11 captures muts [7] from node 5
      stop (ancestors disjoint from [7, 8))
  visit node=10 interval=[7, 8) Mu_rel=0/0 (none) Iu=None
    -> Pruning (root): no relevan

CytoscapeWidget(cytoscape_layout={'name': 'dagre'}, cytoscape_style=[{'selector': 'node', 'style': {'font-fami…

Selected parents for offspring 3: h1=1, h2=0
Generated breakpoints: [3 5 8]
[recombine_multi] offspring=-14 segments=[(0, np.int64(3)), (1, np.int64(5)), (0, np.int64(8)), (1, 11)]
[recombine_multi] segment: parent=0 interval=[0, 3)
[recurse_attach] enter root=0 interval=[0, 3) offspring=-14 gen_v=8
  visit node=0 interval=[0, 3) Mu_rel=2/2 (all) Iu=(8, 11)
    -> Bubble & Strip: bubble 15 captures muts [1, 2] from node 0
      stop (ancestors disjoint from [0, 3))
[recombine_multi] segment: parent=1 interval=[3, 5)
[recurse_attach] enter root=1 interval=[3, 5) offspring=-14 gen_v=9
  visit node=1 interval=[3, 5) Mu_rel=0/1 (none) Iu=(7, 9)
    -> Pruning: dead end at node 1, stop
[recombine_multi] segment: parent=0 interval=[5, 8)
[recurse_attach] enter root=0 interval=[5, 8) offspring=-14 gen_v=10
  visit node=0 interval=[5, 8) Mu_rel=0/2 (none) Iu=(8, 11)
    -> Pruning: dead end at node 0, stop
[recombine_multi] segment: parent=1 interval=[8, 11)
[recurse_attach] enter root=1 inter

...

Selected parents for offspring 1: h1=12, h2=14
Generated breakpoints: [ 3  5 10]
[recombine_multi] offspring=-20 segments=[(12, np.int64(3)), (14, np.int64(5)), (12, np.int64(10)), (14, 11)]
[recombine_multi] segment: parent=12 interval=[0, 3)
[recurse_attach] enter root=12 interval=[0, 3) offspring=-20 gen_v=17
  visit node=12 interval=[0, 3) Mu_rel=0/0 (none) Iu=(3, 11)
    -> Pruning: dead end at node 12, stop
[recombine_multi] segment: parent=14 interval=[3, 5)
[recurse_attach] enter root=14 interval=[3, 5) offspring=-20 gen_v=18
  visit node=14 interval=[3, 5) Mu_rel=0/0 (none) Iu=(1, 9)
    -> Decomposition: bypass node 14, recurse parents [15, 6] with [3, 5)
  visit node=15 interval=[3, 5) Mu_rel=0/1 (none) Iu=(2, 3)
    -> Pruning: dead end at node 15, stop
  visit node=6 interval=[3, 5) Mu_rel=0/1 (none) Iu=None
    -> Pruning (root): no relevant muts on root 6, stop
[recombine_multi] segment: parent=12 interval=[5, 10)
[recurse_attach] enter root=12 interval=[5, 10) offspring

CytoscapeWidget(cytoscape_layout={'name': 'dagre'}, cytoscape_style=[{'selector': 'node', 'style': {'font-fami…

Selected parents for offspring 3: h1=16, h2=8
Generated breakpoints: [3 6 7 9]
[recombine_multi] offspring=-23 segments=[(8, np.int64(3)), (16, np.int64(6)), (8, np.int64(7)), (16, np.int64(9)), (8, 11)]
[recombine_multi] segment: parent=8 interval=[0, 3)
[recurse_attach] enter root=8 interval=[0, 3) offspring=-23 gen_v=25
  visit node=8 interval=[0, 3) Mu_rel=0/0 (none) Iu=(4, 11)
    -> Pruning: dead end at node 8, stop
[recombine_multi] segment: parent=16 interval=[3, 6)
[recurse_attach] enter root=16 interval=[3, 6) offspring=-23 gen_v=26
  visit node=16 interval=[3, 6) Mu_rel=0/0 (none) Iu=(0, 10)
    -> Decomposition: bypass node 16, recurse parents [17, 18, 6, 11, 19] with [3, 6)
  visit node=17 interval=[3, 6) Mu_rel=0/1 (none) Iu=None
    -> Pruning (root): no relevant muts on root 17, stop
  visit node=18 interval=[3, 6) Mu_rel=0/1 (none) Iu=None
    -> Pruning (root): no relevant muts on root 18, stop
  visit node=6 interval=[3, 6) Mu_rel=0/1 (none) Iu=None
    -> Pruning (r

CytoscapeWidget(cytoscape_layout={'name': 'dagre'}, cytoscape_style=[{'selector': 'node', 'style': {'font-fami…

AUDIT 1 -- decision case histogram
case                                  count       %
------------------------------------------------------------
pruning                                  18   12.2%
pruning_root                             59   40.1%
path_compression                          6    4.1%
decomposition                            28   19.0%
direct_attach                             1    0.7%
direct_attach_root                       24   16.3%
direct_attach_dup                         0    0.0%
bubble_strip                              5    3.4%
bubble_split                              3    2.0%
bubble_fill                               0    0.0%
bubble_strip_partial                      0    0.0%
bubble_split_partial                      0    0.0%
bubble_strip_partial_rt                   3    2.0%
------------------------------------------------------------
TOTAL DECISIONS                         147

visits (post-skip-guards)               147
skip_empty_interval       

In [79]:
print(new_offspring)

[20, 21, 23, 25]
